## 第四周第四天——准备大项目！

# The Sidekick（贴身助手）

是时候引入：

1. 结构化输出
2. 多 Agent 流程

In [ ]:
# ========================================================================
# 类型标注 — Python 标准库 typing
# ========================================================================
#from typing import (
#    Annotated,   # 给类型"贴标签"，附加元数据。在 LangGraph 中用来挂 reducer 函数
                 # 例: Annotated[list, add_messages] — "我是 list，更新时用 add_messages 合并"
 #   TypedDict,   # 定义有固定键的字典类型。在 LangGraph 中作为 State 的轻量替代
                 # 例: class State(TypedDict): messages: Annotated[list, add_messages]
#    List,        # 类型标注: List[str] = 字符串列表
#    Dict,        # 类型标注: Dict[str, Any] = 键是 str、值是任意类型的字典
#    Any,         # 类型标注: 任意类型，用在不确定具体类型的地方
#    Optional,    # 类型标注: Optional[str] = 可以是 str 也可以是 None
#)

# ========================================================================
# LangChain 消息对象 — 替代 dict 表示聊天消息
# ========================================================================
#from langchain_core.messages import (
#    AIMessage,       # AI/助手发送的消息
                     # 替代 {"role": "assistant", "content": "..."}
                     # 带有 tool_calls 属性、response_metadata 等丰富信息
#    HumanMessage,    # 用户发送的消息
                     # 替代 {"role": "user", "content": "..."}
#    SystemMessage,   # 系统指令消息，设定 AI 的角色和行为规则
                     # 替代 {"role": "system", "content": "..."}
                     # 不会被用户看到，但 LLM 会遵守里面的规则
#)

# ========================================================================
# LLM 调用 — 统一接口调用各种大模型
# ========================================================================
# from langchain_openai import ChatOpenAI
# ChatOpenAI: 通过 OpenAI 兼容接口调用模型
# 支持 GPT-4o、GPT-4o-mini、DeepSeek 等
# 关键方法:
#   .invoke(messages)            → 同步调用
#   .bind_tools(tools)           → 绑定工具，让 LLM 能调用外部函数
#   .with_structured_output(cls) → 绑定 Pydantic Schema，强制 LLM 返回结构化 JSON

# ========================================================================
# Playwright 浏览器操控 — 让 Agent 操控真实浏览器
# ========================================================================
#from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
# PlayWrightBrowserToolkit: 自动生成浏览器操控工具的工厂
#   .from_browser(browser) → 返回一个 Toolkit 对象
#   .get_tools()           → 返回 [navigate_browser, extract_text, click, ...]

#from langchain_community.tools.playwright.utils import create_async_playwright_browser
# create_async_playwright_browser: 启动一个真实的 Chromium 浏览器实例
#   headless=False → 可见窗口，能看浏览器操作过程
#   headless=True  → 后台静默运行
#   因为是异步的，所有操作都要 await

# ========================================================================
# LangGraph 核心 — Agent 工作流引擎
# ========================================================================
#from langgraph.graph import (
    #StateGraph,  # 工作流图引擎 — 核心类
                 # .add_node("名字", 函数)          → 注册节点
                 # .add_edge("A", "B")              → 固定边 A→B
                 # .add_conditional_edges("A", fn, {  → 条件边
                 #     "结果1": "目标1",
                 #     "结果2": "目标2"
                 # })
                 # .compile(checkpointer=...) → 编译成可执行的图
    #START,       # 图的入口标记 — 对话从这开始
                 # graph_builder.add_edge(START, "第一个节点")
    #END,         # 图的出口标记 — 对话在这结束
                 # graph_builder.add_edge("最后一个节点", END)
#)

# ========================================================================
# Checkpointer — 对话状态持久化
# ========================================================================
#from langgraph.checkpoint.memory import MemorySaver
# MemorySaver: 在内存中保存每次状态变更的快照
#   重启 kernel → 丢失（仅限单次运行）
#   优势: 零配置、快速、适合演示
#   生产替代: SqliteSaver / PostgresSaver

# ========================================================================
# ToolNode — 内置工具执行节点
# ========================================================================
#from langgraph.prebuilt import ToolNode
# ToolNode: LangGraph 内置的工具执行器
#   自动做三件事:
#     ① 检查 LLM 返回的 finish_reason 是否有 tool_calls
#     ② 提取 tool_calls，按名字找到对应函数并执行
#     ③ 把执行结果打包成 tool 角色消息返回
#   不需要自己写 "解析 tool_calls → if/elif 分支 → 调函数" 的代码
#   用法: ToolNode(tools=[tool1, tool2, ...])

# ========================================================================
# add_messages — 消息合并 reducer
# ========================================================================
#from langgraph.graph.message import add_messages
# add_messages: LangGraph 内置的消息合并函数（reducer）
#   作用: 把节点返回的新消息追加到已有消息列表后面
#   更新逻辑: 新列表 = 旧列表 + 追加列表（不是覆盖）
#   用法: messages: Annotated[list, add_messages]
#   没有它 → 每轮对话都会丢失历史

# ========================================================================
# Pydantic — 数据校验与结构化输出
# ========================================================================
#from pydantic import (
#    BaseModel,  # Pydantic 基类，定义数据模型
                # lab4 用它定义 EvaluatorOutput 结构化输出 schema
                # 配合 LLM.with_structured_output() 强制 LLM 返回指定格式 JSON
                # 例:
                #   class EvaluatorOutput(BaseModel):
                #       feedback: str
                #       success_criteria_met: bool
#    Field,      # 给字段添加额外描述和校验规则
                # Field(description="...") → 描述会发给 LLM，告诉它这个字段该填什么
                # Field(default=None)     → 设置默认值
                # 例:
                #   feedback: str = Field(description="Feedback on the response")
#)

# ========================================================================
# 可视化 — notebook 内显示流程图
# ========================================================================
#from IPython.display import (
#    Image,    # 显示图片
              # graph.get_graph().draw_mermaid_png() 返回一张 PNG 图片
              # Image(...) 把它嵌入到 notebook 输出区
#    display,  # 在 notebook cell 中渲染输出
              # display(Image(...)) 会在 cell 下方直接显示流程图
#)

In [2]:
from typing import Annotated, TypedDict, List, Dict, Any, Optional
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_async_playwright_browser
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from IPython.display import Image, display
import gradio as gr
import uuid
from dotenv import load_dotenv

In [3]:
load_dotenv(override=True)

True

### 对于结构化输出，我们定义一个 Pydantic 对象作为 Schema

In [4]:
# 首先定义结构化输出
# 为什么定义EvaluatorOutput(BaseModel)是用BaseModel而不是TypedDict
# 1.先问自己定义的agent的作用是什么？是输出固定格式内容还是固定message内容？
# 如果是前者，BaseModel更合适；如果是后者，TypedDict可能更简洁
# BaseModel有强类型限制。必须是定义的类型才能处理。
# 在定义 State 时，如果你只是做 Agent 内部上下文流转，用 TypedDict 更快；
# 如果要保证 数据一致性和安全性，用 BaseModel 更合适。
# 这边因为是做评估器，输出是固定格式内容，所以用BaseModel更合适
# BaseModel定义需要输出的字段，类型，描述
class EvaluatorOutput(BaseModel):#评估器
    #反馈
    feedback: str = Field(description="Feedback on the assistant's response")
    #成功标准已达成
    success_criteria_met: bool = Field(description="Whether the success criteria have been met")
    #需要用户输入
    user_input_needed: bool = Field(description="True if more input is needed from the user, or clarifications, or the assistant is stuck")

In [5]:
# 如何设计一个合格的Schema(模式)
# 对 BaseModel（给 LLM 的结构化输出）
#□ 消费方是谁？每个字段都被读了吗？
#□ 每个字段的 description 足够清晰吗？（LLM 只能靠这个理解字段含义）
#□ 字段类型越简单越好：bool > Enum > int > str
#□ 避免让 LLM 做模糊判断，能量化就量化
#□ 有没有冗余字段？（定义了很多但没人读）

# 对于 对 TypedDict（给 LangGraph 的 State）
#□ 这个字段需要跨节点共享吗？（不需要 → 放节点局部变量里）
#□ 用追加还是覆盖？
#    - 对话/日志/事件 → 追加（Annotated + reducer）
#    - 状态/开关/配置 → 覆盖
#□ 类型标注准确吗？Optional 用对了吗？
#□ 初始值合理吗？（False / None / "" / []）

# 思考:评估agent本质上是判断和处理工作agent是否达标的判断器。
# 1.对于用户输入的数据。存在正确能处理的数据--错误但能处理的数据--错误且无法处理的数据三种情况。评估器需要对这三种情况都做出正确的判断和处理。
# 对于评估器的输出结果。存在正确评估结果--错误但能处理的评估结果--错误且无法处理的评估结果三种情况。评估器需要对这三种情况都做出正确的判断和处理。
# 其实两个字段就够了反馈和判断是否完成，
# 因为反馈是文本大模型无法更好处理所以需要多添加一个
# 需要用户输入的字段来让模型判断是否需要用户输入来辅助完成任务。


### State 我们继续使用 TypedDict

但现在我们需要维护一些真正的信息了！

`messages` 使用 reducer 来合并。其他字段则是普通值，每次状态变更时直接覆盖。

In [6]:
# State 定义
# 内部流转时用TypedDict更快；如果要保证 数据一致性和安全性，用 BaseModel 更合适。
class State(TypedDict):
    messages: Annotated[List[Any], add_messages]
    success_criteria: str #评判标准。本来由ai生成的，这边由用户输入。
    #作为特别说明的重要数据需要在state中流转。
    feedback_on_work: Optional[str] #对工作的反馈，评估器给出的文本反馈，
    #告诉工作agent哪里做得好哪里需要改进,可以没有
    success_criteria_met: bool
    user_input_needed: bool

In [ ]:
# 获取我们的异步 Playwright 工具
# 如果在这里或后续步骤中遇到 NotImplementedError，请参考 3_lab3 notebook 顶部的"注意"说明
import nest_asyncio
nest_asyncio.apply()

# Windows 上 Playwright 需要 SelectorEventLoop（Proactor 不支持 subprocess）
import sys
import asyncio
if sys.platform.startswith("win"):
    from asyncio import WindowsSelectorEventLoopPolicy
    asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())

async_browser =  create_async_playwright_browser(headless=False)  # headful 模式（可见浏览器窗口）
toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=async_browser)
tools = toolkit.get_tools()

In [ ]:
# ============================================================
# 初始化两个 LLM — 分别使用不同的模型提供商
# ============================================================

# ——— Worker LLM：DeepSeek（负责执行任务 + 调用浏览器工具）———
worker_llm = ChatOpenAI(
    model="deepseek-chat",
    base_url="https://api.deepseek.com",
    api_key="sk-1784e2115de34c99984acfbbb8ed8dd8",
)
worker_llm_with_tools = worker_llm.bind_tools(tools)

# ——— Evaluator LLM：MiniMax（负责评估 Worker 输出是否达标）———
evaluator_llm = ChatOpenAI(
    model="MiniMax-Text-01",
    base_url="https://api.minimax.chat/v1",
    api_key="sk-ct6ct1y17ry3m9xh2rce3bbx68kbsqs19y326ym89hxw2k64",
)
# 这里定义的输出大模型必须按EvaluatorOutput输出。调用evaluator_llm_with_output后
evaluator_llm_with_output = evaluator_llm.with_structured_output(EvaluatorOutput)

In [ ]:
# worker 节点：执行任务的 Agent
# 为什么返回的是 Dict[str, Any] 而不是 State？
#Dict[str, Any] 是 LangGraph 节点最常用的返回类型——"我只告诉你哪些字段变了，剩下的你帮我保留"。
# Reducer 自动处理追加/覆盖逻辑，不需要返回完整 State。
# {state['success_criteria']}把 state 里的"success_criteria"取出来并插入到 system_message 字符串里
def worker(state: State) -> Dict[str, Any]:
    system_message = f"""You are a helpful assistant that can use tools to complete tasks.
You keep working on a task until either you have a question or clarification for the user, or the success criteria is met.
This is the success criteria:
{state['success_criteria']}
You should reply either with a question for the user about this assignment, or with your final response.
If you have a question for the user, you need to reply by clearly stating your question. An example might be:

Question: please clarify whether you want a summary or a detailed answer

If you've finished, reply with the final answer, and don't ask a question; simply reply with the answer.
"""
    #这边是获取评估器的反馈并注入 system message 里，
    # 让工作 agent 知道哪里需要改进，继续完成任务
    if state.get("feedback_on_work"):
        system_message += f"""
Previously you thought you completed the assignment, but your reply was rejected because the success criteria was not met.
Here is the feedback on why this was rejected:
{state['feedback_on_work']}
With this feedback, please continue the assignment, ensuring that you meet the success criteria or have a question for the user."""
    
    # 注入 system message
    # found_system_message = False 为什么要这么设计呢？
    # 这里的worker是一个工作节点，是无状态的。
    # 只有在重试时才会追加system_message。好比生成新的提示词(提示词优化)
    found_system_message = False
    messages = state["messages"] #这里创建了一个新的变量来存储 state 里的消息列表，方便后续操作
    for message in messages:
        if isinstance(message, SystemMessage):
            message.content = system_message
            found_system_message = True
    
    if not found_system_message:
        messages = [SystemMessage(content=system_message)] + messages
    
    # 调用带工具的 LLM
    # 根据messages请求决策处理
    # invoke方法会把消息列表发给 LLM，LLM 
    # 根据 system_message 里的规则和工具调用结果生成回复
    # 调用工具大模型
    response = worker_llm_with_tools.invoke(messages)
    
    # 返回更新后的 state
    # 每次决定要调用工具也会加入信息，而不是单纯的对话内容
    return {
        "messages": [response],
    }

In [ ]:
# worker 路由器：判断 worker 输出后该走 tools 还是 evaluator
def worker_router(state: State) -> str:
    last_message = state["messages"][-1]
    # hasattr 双重校验
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    else:
        return "evaluator"

In [ ]:
# 格式化对话历史，供 evaluator 评估时查看
# 给agent的数据都需要格式化成字符串，方便评估器理解和判断
def format_conversation(messages: List[Any]) -> str:
    conversation = "Conversation history:\n\n"
    for message in messages:
        if isinstance(message, HumanMessage):
            conversation += f"User: {message.content}\n"
            # 工具使用的消息没有 content，只有 tool_calls，所以用占位符表示
        elif isinstance(message, AIMessage):
            text = message.content or "[Tools use]"
            conversation += f"Assistant: {text}\n"
    return conversation

In [ ]:
# evaluator 节点：评估 worker 的输出是否达标

def evaluator(state: State) -> State:
    # py的语法糖[-1]就是最后
    last_response = state["messages"][-1].content

    system_message = """You are an evaluator that determines if a task has been completed successfully by an Assistant.
Assess the Assistant's last response based on the given criteria. Respond with your feedback, and with your decision on whether the success criteria has been met,
and whether more input is needed from the user."""
    
    user_message = f"""You are evaluating a conversation between the User and Assistant. You decide what action to take based on the last response from the Assistant.

The entire conversation with the assistant, with the user's original request and all replies, is:
{format_conversation(state['messages'])}

The success criteria for this assignment is:
{state['success_criteria']}

And the final response from the Assistant that you are evaluating is:
{last_response}

Respond with your feedback, and decide if the success criteria is met by this response.
Also, decide if more user input is required, either because the assistant has a question, needs clarification, or seems to be stuck and unable to answer without help.
"""
    if state["feedback_on_work"]:
        user_message += f"Also, note that in a prior attempt from the Assistant, you provided this feedback: {state['feedback_on_work']}\n"
        user_message += "If you're seeing the Assistant repeating the same mistakes, then consider responding that user input is required."
    #为什么有system_message和user_message?
    #这是 LLM API 调用里的一个通用设计模式。
    # 对于每次请求system的背景设定一定是不变的。要处理的内容是动态的
    evaluator_messages = [SystemMessage(content=system_message), HumanMessage(content=user_message)]
    # 调用输出大模型
    # 输出的内容可能是不合格的后面有评估路由处理。
    eval_result = evaluator_llm_with_output.invoke(evaluator_messages)
    new_state = {
        "messages": [{"role": "assistant", "content": f"Evaluator Feedback on this answer: {eval_result.feedback}"}],
        # 在
        "feedback_on_work": eval_result.feedback,
        "success_criteria_met": eval_result.success_criteria_met,
        "user_input_needed": eval_result.user_input_needed
    }
    return new_state

In [ ]:
# 评估结果路由器：达标或需用户输入 → 结束，否则 → 返回 worker 重试
def route_based_on_evaluation(state: State) -> str:
    if state["success_criteria_met"] or state["user_input_needed"]:
        return "END"
    else:
        return "worker"

In [ ]:
# 用 State 初始化 Graph Builder
graph_builder = StateGraph(State)

# 添加三个节点
graph_builder.add_node("worker", worker)
graph_builder.add_node("tools", ToolNode(tools=tools))
graph_builder.add_node("evaluator", evaluator)

# 添加边
# worker → 条件路由（调工具 / 去评估）
# 这里是流程  worker处理-路由处理-两种结果调用工具或者进行评估
# 条件边使用add_conditional_edges
graph_builder.add_conditional_edges("worker", worker_router, {"tools": "tools", "evaluator": "evaluator"})
# 工具执行完后 → 回到 worker 继续
graph_builder.add_edge("tools", "worker")
# evaluator → 条件路由（重试 / 结束）
graph_builder.add_conditional_edges("evaluator", route_based_on_evaluation, {"worker": "worker", "END": END})
# 从 worker 开始
graph_builder.add_edge(START, "worker")

# 编译 graph，传入 checkpointer 持久化记忆
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

### 接下来是 Gradio 回调，用于触发一个 super-step

In [ ]:
# 生成唯一线程 ID
def make_thread_id() -> str:
    return str(uuid.uuid4())

# 处理用户消息：构建 state → 调用 graph → 返回结果
async def process_message(message, success_criteria, history, thread):

    config = {"configurable": {"thread_id": thread}}

    state = {
        "messages": message,
        "success_criteria": success_criteria,
        "feedback_on_work": None,
        "success_criteria_met": False,
        "user_input_needed": False
    }
    result = await graph.ainvoke(state, config=config)
    user = {"role": "user", "content": message}
    reply = {"role": "assistant", "content": result["messages"][-2].content}
    feedback = {"role": "assistant", "content": result["messages"][-1].content}
    return history + [user, reply, feedback]

# 重置会话状态
async def reset():
    return "", "", None, make_thread_id()


### 现在启动我们的 Sidekick 界面

In [ ]:

with gr.Blocks(theme=gr.themes.Default(primary_hue="emerald")) as demo:
    gr.Markdown("## Sidekick Personal Co-worker")
    thread = gr.State(make_thread_id())
    
    with gr.Row():
        chatbot = gr.Chatbot(label="Sidekick", height=300, type="messages")
    with gr.Group():
        with gr.Row():
            message = gr.Textbox(show_label=False, placeholder="Your request to your sidekick")
        with gr.Row():
            success_criteria = gr.Textbox(show_label=False, placeholder="What are your success critiera?")
    with gr.Row():
        reset_button = gr.Button("Reset", variant="stop")
        go_button = gr.Button("Go!", variant="primary")
    message.submit(process_message, [message, success_criteria, chatbot, thread], [chatbot])
    success_criteria.submit(process_message, [message, success_criteria, chatbot, thread], [chatbot])
    go_button.click(process_message, [message, success_criteria, chatbot, thread], [chatbot])
    reset_button.click(reset, [], [message, success_criteria, chatbot, thread])

    
demo.launch()

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">恭喜你完成了 Sidekick 的第一个版本！</h2>
            <span style="color:#00cc00;">这是课程中一个非常精彩的时刻。你创造了一个非常强大工具的起点。而且你已经掌握了一个令人印象深刻的 Agent 框架——LangGraph。也许你和我一样，正在从一个 LangGraph 怀疑者转变为 LangGraph 粉丝..<br/><br/>如果我不再次提到：如果你能在 Udemy 上给课程评分，我将非常感激：这是 Udemy 决定是否向其他人展示课程的主要方式，影响非常巨大。<br/><br/>还有另一个提醒，如果你还没有在 <a href="https://www.linkedin.com/in/eddonner/">LinkedIn 上联系我</a>，我非常乐意！如果你想发布你的课程进度，请标记我，我会参与互动来增加你的曝光度。
            </span>
        </td>
    </tr>